# **Libraries**

In [ ]:
%run nb_spn_common

# **Functions**

## **Creation functions**

In [ ]:
# --------------------------------------------------------
#  Workspace creation function
# --------------------------------------------------------
def create_workspace(
        access_token: str
      , workspace_name: str
      , capacity_id: str = None
) -> str:
    """
    Create a Microsoft Fabric Workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param workspace_name: Display name for the Workspace.
    :param capacity_id: Optional Fabric capacity GUID for assignment.
    :returns: Workspace GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    endpoint = "https://api.fabric.microsoft.com/v1/workspaces"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {"displayName": workspace_name}
    # Capacity assignment is optional - unassigned workspaces have no capacity
    if capacity_id:
        body["capacityId"] = capacity_id

    response = requests.post(endpoint, headers=headers, json=body)

    # 201 = synchronous creation complete, 202 = LRO in progress
    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Workspace '{workspace_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Lakehouse creation function
# --------------------------------------------------------
def create_lakehouse(
        access_token: str
      , lakehouse_name: str
      , enable_schemas: bool = False
) -> str:
    """
    Create a Microsoft Fabric Lakehouse in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param lakehouse_name: Display name for the Lakehouse.
    :param enable_schemas: When True, provisions a schema-enabled Lakehouse.
    :returns: Lakehouse GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {"displayName": lakehouse_name}
    if enable_schemas:
        # Preview flag - only True is valid on this payload
        body["creationPayload"] = {"enableSchemas": True}

    response = requests.post(endpoint, headers=headers, json=body)

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Lakehouse '{lakehouse_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Notebook creation function
# --------------------------------------------------------
def create_notebook(access_token: str, notebook_name: str) -> str:
    """
    Create an empty Fabric Notebook in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param notebook_name: Display name for the Notebook.
    :returns: Notebook GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/notebooks"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": notebook_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Notebook '{notebook_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Warehouse creation function
# --------------------------------------------------------
def create_warehouse(access_token: str, warehouse_name: str) -> str:
    """
    Create a Fabric Warehouse in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param warehouse_name: Display name for the Warehouse.
    :returns: Warehouse GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/warehouses"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": warehouse_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Warehouse '{warehouse_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  SQL Database creation function
# --------------------------------------------------------
def create_sql_server(access_token: str, sql_server_name: str) -> str:
    """
    Create a Fabric SQL Database in the current workspace.

    Uses the generic /items endpoint with type=SQLDatabase rather than a
    type-specific collection - as of writing this is the documented path.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param sql_server_name: Display name for the SQL Database.
    :returns: SQL Database GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    body = {
        "displayName": sql_server_name,
        "type": "SQLDatabase"
    }

    response = requests.post(endpoint, headers=headers, json=body)

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"SQL Database '{sql_server_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Variable library creation function
# --------------------------------------------------------
def create_variable_library(access_token: str, library_name: str) -> str:
    """
    Create a Microsoft Fabric Variable Library in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param library_name: Display name for the Variable Library.
    :returns: Variable Library GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/VariableLibraries"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": library_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Variable Library '{library_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Pipeline creation function
# --------------------------------------------------------
def create_pipeline(access_token: str, pipeline_name: str) -> str:
    """
    Create a Fabric Data Pipeline in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param pipeline_name: Display name for the Data Pipeline.
    :returns: Data Pipeline GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/dataPipelines"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": pipeline_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Pipeline '{pipeline_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Dataflow creation function
# --------------------------------------------------------
def create_dataflow_gen2(access_token: str, dataflow_name: str) -> str:
    """
    Create a Microsoft Fabric Dataflow (Gen2) in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param dataflow_name: Display name for the Dataflow.
    :returns: Dataflow GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/dataflows"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": dataflow_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Dataflow Gen2 '{dataflow_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Eventhouse creation function
# --------------------------------------------------------
def create_eventhouse(access_token: str, eventhouse_name: str) -> str:
    """
    Create a Fabric Eventhouse in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param eventhouse_name: Display name for the Eventhouse.
    :returns: Eventhouse GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventhouses"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": eventhouse_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Eventhouse '{eventhouse_name}' created (id={item_id}) by: {client_name}")
    return item_id

# --------------------------------------------------------
#  Eventstream creation function
# --------------------------------------------------------
def create_eventstream(access_token: str, eventstream_name: str) -> str:
    """
    Create a Fabric Eventstream in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param eventstream_name: Display name for the Eventstream.
    :returns: Eventstream GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/eventstreams"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": eventstream_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Eventstream '{eventstream_name}' created (id={item_id}) by: {client_name}")
    return item_id


# --------------------------------------------------------
#  Ontology creation function
# --------------------------------------------------------
def create_ontology(access_token: str, ontology_name: str) -> str:
    """
    Create a Fabric Ontology in the current workspace.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param ontology_name: Display name for the Ontology.
    :returns: Ontology GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/ontologies"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": ontology_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Ontology '{ontology_name}' created (id={item_id}) by: {client_name}")
    return item_id


# --------------------------------------------------------
#  Reflex (Activator) creation function
# --------------------------------------------------------
def create_reflex(access_token: str, reflex_name: str) -> str:
    """
    Create a Fabric Reflex (Activator) in the current workspace.

    NOTE: The /reflexes endpoint was in preview at the time this function was
    written. Confirm current GA status in the Fabric REST API docs before
    relying on it in production automation.

    :param access_token: OAuth2 bearer token (audience: api.fabric.microsoft.com).
    :param reflex_name: Display name for the Reflex.
    :returns: Reflex GUID.
    :raises requests.HTTPError: If the creation request fails.
    :raises RuntimeError: If the LRO fails or returns no ID.
    :raises TimeoutError: If the LRO does not complete in time.
    """
    workspace_id = notebookutils.runtime.context["currentWorkspaceId"]
    endpoint = f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/reflexes"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    response = requests.post(endpoint, headers=headers, json={"displayName": reflex_name})

    if response.status_code == 201:
        item_id = response.json()["id"]
    elif response.status_code == 202:
        result_response = poll_lro(response, headers, return_result=True)
        item_id = result_response.json()["id"]
    else:
        response.raise_for_status()

    print(f"Reflex '{reflex_name}' created (id={item_id}) by: {client_name}")
    return item_id

# **Operation**

### **Create single Fabric item**

In [ ]:
# --------------------------------------------------------
#  Create Workspace with Service Principal
# --------------------------------------------------------
workspace_name = '<WorkspaceName>'
workspace_id = create_workspace(access_token, workspace_name, capacity_id)

# --------------------------------------------------------
#  Create Lakehouse with Service Principal
# --------------------------------------------------------
lakehouse_name = '<LakehouseName>'
lakehouse_id = create_lakehouse(access_token, lakehouse_name, True) # True for schema-enabled

# --------------------------------------------------------
#  Create Notebook with Service Principal
# --------------------------------------------------------
notebook_name = '<NotebookName>'
notebook_id = create_notebook(access_token, notebook_name)

# --------------------------------------------------------
#  Create Warehouse with Service Principal
# --------------------------------------------------------
warehouse_name = '<WarehouseName>'
warehouse_id = create_warehouse(access_token, warehouse_name)

# --------------------------------------------------------
#  Create Pipeline with Service Principal
# --------------------------------------------------------
pipeline_name = '<PipelineName>'
pipeline_id = create_pipeline(access_token, pipeline_name)

# --------------------------------------------------------
#  Create Dataflow with Service Principal
# --------------------------------------------------------
dataflow_name = '<DataflowName>'
dataflow_id = create_dataflow_gen2(access_token, dataflow_name)

# --------------------------------------------------------
#  Create SQL Database with Service Principal
# --------------------------------------------------------
sql_server_name = '<SQLServerName>'
sql_server_id = create_sql_server(access_token, sql_server_name)

# --------------------------------------------------------
#  Create Variable Library with Service Principal
# --------------------------------------------------------
variable_library_name = '<VariableLibraryName>'
variable_library_id = create_variable_library(access_token, variable_library_name)

# --------------------------------------------------------
#  Create Eventstream with Service Principal
# --------------------------------------------------------
eventstream_name = '<EventstreamName>'
eventstream_id = create_eventstream(access_token, eventstream_name)

# --------------------------------------------------------
#  Create Eventhouse with Service Principal
# --------------------------------------------------------
eventhouse_name = '<EventhouseName>'
eventhouse_id = create_eventhouse(access_token, eventhouse_name)

# --------------------------------------------------------
#  Create Ontology with Service Principal
# --------------------------------------------------------
ontology_name = '<OntologyName>'
ontology_id = create_ontology(access_token, ontology_name)

# --------------------------------------------------------
#  Create Reflex (Activator) with Service Principal
# --------------------------------------------------------
reflex_name = '<ReflexName>'
reflex_id = create_reflex(access_token, reflex_name)

### **Create multiple Fabric items**

In [ ]:
# --------------------------------------------------------
#  Multi-item creation - Tier 2 (best-effort per item)
# --------------------------------------------------------
# STRICT controls what happens when the loop finishes with failures:
#   False (default) - print the summary and continue. Most automation wants
#                     best-effort with a clear report at the end.
#   True  - raise RuntimeError after the summary to fail the notebook run.
STRICT = False

# --------------------------------------------------------
#  Define creators for multi item deployment
# --------------------------------------------------------
creators = {
    "lakehouse": lambda token, name: create_lakehouse(token, name, True),
    "notebook": create_notebook,
    "warehouse": create_warehouse,
    "pipeline": create_pipeline,
    "dataflow": create_dataflow_gen2,
    "variable_library": create_variable_library,
}

# --------------------------------------------------------
#  Create list of items by type
# --------------------------------------------------------
items_to_create = [
    ("lakehouse", ["<LakehouseName1>", "<LakehouseName2>"]),
    ("notebook", ["<NotebookName1>", "<NotebookName2>", "<NotebookName3>"]),
    ("warehouse", ["<WarehouseName>"]),
    ("pipeline", ["<PipelineName1>", "<PipelineName2>", "<PipelineName3>"]),
    ("dataflow", ["<DataflowName1>", "<DataflowName2>"]),
    ("variable_library", ["<VariableLibraryName1>", "<VariableLibraryName2>"]),
]

# --------------------------------------------------------
#  Loop through lists and create defined items
# --------------------------------------------------------
# created_ids holds the GUIDs produced by each successful create_* call, keyed
# by (type, name). Downstream cells can look up by key to chain operations
# (e.g. binding a pipeline to a new lakehouse) without re-querying Fabric.
results = {
    "succeeded": [],  # list[str]: "<type>/<name>"
    "skipped":   [],  # list[dict]: {"name": str, "reason": str}
    "failed":    [],  # list[dict]: {"name": str, "error": str}
}
created_ids: Dict = {}

for item_type, names in items_to_create:
    for name in names:
        key = f"{item_type}/{name}"

        # Skip unfilled placeholders so the template notebook doesn't error
        # on first run before the user has customized the lists.
        if name.startswith("<") and name.endswith(">"):
            results["skipped"].append({"name": key, "reason": "placeholder not filled"})
            continue

        try:
            item_id = creators[item_type](access_token, name)
            created_ids[(item_type, name)] = item_id
            results["succeeded"].append(key)
        except Exception as ex:
            results["failed"].append({"name": key, "error": str(ex)})

# --------------------------------------------------------
#  Summary
# --------------------------------------------------------
print("\n" + "─" * 46)
print("  Summary")
print("─" * 46)
print(f"  Succeeded: {len(results['succeeded'])}")
print(f"  Skipped:   {len(results['skipped'])}")
print(f"  Failed:    {len(results['failed'])}")

for entry in results["failed"]:
    print(f"    - {entry['name']}: {entry['error']}")
for entry in results["skipped"]:
    print(f"    ~ {entry['name']}: {entry['reason']}")

if STRICT and results["failed"]:
    raise RuntimeError(
        f"STRICT mode: {len(results['failed'])} item(s) failed. "
        f"See summary above."
    )